# Dataset loading

This section loads the local Polish road sign classification dataset and checks the number of images in each class.

In [ ]:
from pathlib import Path
import math
import random
import shutil

import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
PROJECT_ROOT = next(
    (
        path for path in [Path.cwd(), *Path.cwd().parents]
        if (path / "data" / "classification").exists()
    ),
    None,
)

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Could not find data/classification. "
        "Open this notebook from the project folder or from notebooks/."
    )

DATA_DIR = PROJECT_ROOT / "data"
CLASSIFICATION_DIR = DATA_DIR / "classification"

CLASSIFICATION_DIR.exists()


In [ ]:
class_dirs = sorted([path for path in CLASSIFICATION_DIR.iterdir() if path.is_dir()])
class_names = [path.name for path in class_dirs]

class_names


In [ ]:
image_extensions = {".jpg", ".jpeg", ".png"}

class_counts = []

for class_dir in class_dirs:
    image_count = len([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    class_counts.append({
        "class_name": class_dir.name,
        "image_count": image_count,
    })

df_classes = pd.DataFrame(class_counts)
df_classes


In [ ]:
total_images = df_classes["image_count"].sum()
total_images


# Exploratory data analysis

This section checks the basic structure of the classification dataset.

In [ ]:
num_classes = len(df_classes)
num_classes


In [ ]:
df_classes.sort_values("image_count").head()


In [ ]:
df_classes.sort_values("image_count", ascending=False).head()


In [ ]:
plt.figure(figsize=(12, 6))
plt.bar(df_classes["class_name"], df_classes["image_count"])
plt.xticks(rotation=90)
plt.xlabel("Class")
plt.ylabel("Number of images")
plt.title("Number of images per class")
plt.show()


# Sample road sign images

This section displays example images from selected road sign classes.

In [ ]:
sample_images = []

for class_dir in class_dirs:
    image_paths = sorted([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    if image_paths:
        sample_images.append({
            "class_name": class_dir.name,
            "image_path": image_paths[0],
        })

len(sample_images)


In [ ]:
columns = 4
rows = math.ceil(len(sample_images) / columns)

fig, axes = plt.subplots(rows, columns, figsize=(12, rows * 3))
axes = axes.flatten()

for ax, sample in zip(axes, sample_images):
    image = plt.imread(sample["image_path"])
    ax.imshow(image)
    ax.set_title(sample["class_name"])
    ax.axis("off")

for ax in axes[len(sample_images):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


# Data preprocessing

This section prepares the classification dataset split for YOLO11 training.

In [ ]:
PROCESSED_DATA_DIR = DATA_DIR / "processed" / "yolo11_classification"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42

TRAIN_RATIO + VAL_RATIO + TEST_RATIO


In [ ]:
image_records = []

for class_dir in class_dirs:
    image_paths = sorted([
        path for path in class_dir.rglob("*")
        if path.suffix.lower() in image_extensions
    ])

    for image_path in image_paths:
        image_records.append({
            "class_name": class_dir.name,
            "image_path": image_path,
        })

df_images = pd.DataFrame(image_records)
df_images.head()


In [ ]:
rng = random.Random(RANDOM_SEED)
split_records = []

for class_name, group in df_images.groupby("class_name"):
    image_paths = group["image_path"].tolist()
    rng.shuffle(image_paths)

    total_count = len(image_paths)
    train_count = int(total_count * TRAIN_RATIO)
    val_count = int(total_count * VAL_RATIO)

    train_paths = image_paths[:train_count]
    val_paths = image_paths[train_count:train_count + val_count]
    test_paths = image_paths[train_count + val_count:]

    for split_name, paths in [
        ("train", train_paths),
        ("val", val_paths),
        ("test", test_paths),
    ]:
        for image_path in paths:
            split_records.append({
                "split": split_name,
                "class_name": class_name,
                "image_path": image_path,
            })

df_split = pd.DataFrame(split_records)
df_split.head()


In [ ]:
split_summary = (
    df_split
    .groupby(["split", "class_name"])
    .size()
    .reset_index(name="image_count")
)

split_summary.head()


In [ ]:
df_split["split"].value_counts()


# Przygotowanie danych dla YOLO11

Ta cz??? tworzy struktur? katalog?w wymagan? przez klasyfikacj? YOLO: `train`, `val` i `test`. Ka?dy z tych katalog?w zawiera podfoldery z nazwami klas.


In [ ]:
def prepare_yolo11_classification_dataset(df_split, output_dir, overwrite=False):
    output_dir = Path(output_dir)

    if overwrite and output_dir.exists():
        shutil.rmtree(output_dir)

    output_dir.mkdir(parents=True, exist_ok=True)

    for row in df_split.itertuples(index=False):
        destination_dir = output_dir / row.split / row.class_name
        destination_dir.mkdir(parents=True, exist_ok=True)

        destination_path = destination_dir / row.image_path.name
        shutil.copy2(row.image_path, destination_path)

    return output_dir


In [ ]:
PREPARE_DATASET = True
OVERWRITE_DATASET = False

if PREPARE_DATASET:
    prepare_yolo11_classification_dataset(
        df_split=df_split,
        output_dir=PROCESSED_DATA_DIR,
        overwrite=OVERWRITE_DATASET,
    )

required_split_dirs = [
    PROCESSED_DATA_DIR / "train",
    PROCESSED_DATA_DIR / "val",
    PROCESSED_DATA_DIR / "test",
]

[path.exists() for path in required_split_dirs]


In [ ]:
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

split_summary_path = REPORTS_DIR / "dataset_split_summary.csv"
split_summary.to_csv(split_summary_path, index=False)

split_summary_path


# Trening YOLO11

Domy?lnie trenowany jest tylko `yolo11n-cls.pt`, bo jest najmniejszy i najprostszy do uruchomienia. Do por?wnania modeli mo?na ustawi? `COMPARE_MODELS = True`, wtedy notebook uruchomi te? `yolo11s-cls.pt` i `yolo11m-cls.pt`.


In [ ]:
try:
    from ultralytics import YOLO
except ModuleNotFoundError as exc:
    if exc.name == "ultralytics":
        raise ModuleNotFoundError(
            "Brakuje biblioteki ultralytics w aktywnym kernelu. "
            "Uruchom w notebooku: %pip install -r ../requirements.txt"
        ) from exc
    raise


In [ ]:
YOLO_DATA_DIR = PROCESSED_DATA_DIR
TRAINING_RUNS_DIR = PROJECT_ROOT / "runs" / "yolo11_classification"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

COMPARE_MODELS = False
YOLO_MODELS = ["yolo11n-cls.pt"]

if COMPARE_MODELS:
    YOLO_MODELS = [
        "yolo11n-cls.pt",
        "yolo11s-cls.pt",
        "yolo11m-cls.pt",
    ]

IMAGE_SIZE = 224
EPOCHS = 25
BATCH_SIZE = 32
DEVICE = None  # ustaw "cpu", je?li chcesz wymusi? CPU, albo 0 dla GPU

YOLO_MODELS


In [ ]:
RUN_TRAINING = False

training_results = {}

if RUN_TRAINING:
    for model_name in YOLO_MODELS:
        model = YOLO(model_name)
        run_name = model_name.replace("-cls.pt", "")

        train_kwargs = {
            "data": str(YOLO_DATA_DIR),
            "epochs": EPOCHS,
            "imgsz": IMAGE_SIZE,
            "batch": BATCH_SIZE,
            "project": str(TRAINING_RUNS_DIR),
            "name": run_name,
            "exist_ok": True,
        }
        if DEVICE is not None:
            train_kwargs["device"] = DEVICE

        results = model.train(**train_kwargs)
        training_results[model_name] = results

training_results


# Por?wnanie wynik?w i zapis modelu

Po treningu Ultralytics zapisuje wyniki w `runs/yolo11_classification`. Poni?sze kom?rki zbieraj? ko?cowe metryki z `results.csv` i kopiuj? najlepsze wagi do katalogu `models`.


In [ ]:
def read_last_metrics(run_dir):
    results_csv = run_dir / "results.csv"
    if not results_csv.exists():
        return None

    metrics = pd.read_csv(results_csv)
    if metrics.empty:
        return None

    last_row = metrics.tail(1).copy()
    last_row.insert(0, "run", run_dir.name)
    return last_row

metrics_frames = []

for model_name in YOLO_MODELS:
    run_name = model_name.replace("-cls.pt", "")
    run_dir = TRAINING_RUNS_DIR / run_name
    last_metrics = read_last_metrics(run_dir)

    if last_metrics is not None:
        metrics_frames.append(last_metrics)

if metrics_frames:
    model_comparison = pd.concat(metrics_frames, ignore_index=True)
    model_comparison.to_csv(REPORTS_DIR / "training_summary.csv", index=False)
else:
    model_comparison = pd.DataFrame()

model_comparison


In [ ]:
best_model_paths = []

for model_name in YOLO_MODELS:
    run_name = model_name.replace("-cls.pt", "")
    best_model_path = TRAINING_RUNS_DIR / run_name / "weights" / "best.pt"

    if best_model_path.exists():
        saved_model_path = MODELS_DIR / f"{run_name}-best.pt"
        shutil.copy2(best_model_path, saved_model_path)
        best_model_paths.append(saved_model_path)

best_model_paths


# Walidacja i macierz pomy?ek

Ta cz??? uruchamia walidacj? najlepszego modelu. Ultralytics zapisuje wykresy i macierz pomy?ek w katalogu `runs`.


In [ ]:
RUN_VALIDATION = False
MODEL_TO_VALIDATE = MODELS_DIR / "yolo11n-best.pt"

validation_results = None

if RUN_VALIDATION:
    if not MODEL_TO_VALIDATE.exists():
        raise FileNotFoundError(f"Nie znaleziono modelu: {MODEL_TO_VALIDATE}")

    validation_model = YOLO(str(MODEL_TO_VALIDATE))
    validation_results = validation_model.val(
        data=str(YOLO_DATA_DIR),
        imgsz=IMAGE_SIZE,
        project=str(TRAINING_RUNS_DIR),
        name="validation",
        plots=True,
    )

validation_results


# Predykcja na nowym zdj?ciu

Po treningu mo?na poda? ?cie?k? do dowolnego zdj?cia znaku. Notebook wypisze przewidzian? klas? i pewno?? modelu.


In [ ]:
def predict_road_sign(model_path, image_path):
    model_path = Path(model_path)
    image_path = Path(image_path)

    if not model_path.exists():
        raise FileNotFoundError(f"Nie znaleziono modelu: {model_path}")
    if not image_path.exists():
        raise FileNotFoundError(f"Nie znaleziono zdj?cia: {image_path}")

    model = YOLO(str(model_path))
    result = model.predict(str(image_path), verbose=False)[0]

    class_id = int(result.probs.top1)
    confidence = float(result.probs.top1conf)
    class_name = result.names[class_id]

    return {
        "class_name": class_name,
        "confidence": confidence,
        "image_path": image_path,
    }


In [ ]:
RUN_PREDICTION = False
MODEL_TO_USE = MODELS_DIR / "yolo11n-best.pt"
IMAGE_TO_PREDICT = class_dirs[0] / sorted(class_dirs[0].glob("*.jpg"))[0].name

if RUN_PREDICTION:
    prediction = predict_road_sign(MODEL_TO_USE, IMAGE_TO_PREDICT)
else:
    prediction = None

prediction


# Wnioski

Notebook realizuje pe?ny, prosty pipeline: analiza danych, podzia? na zbiory, przygotowanie katalog?w YOLO11, trening, por?wnanie modeli, walidacja oraz predykcja dla pojedynczego obrazu. Domy?lnie kosztowne kroki s? wy??czone flagami `RUN_TRAINING`, `RUN_VALIDATION` i `RUN_PREDICTION`, ?eby notebook mo?na by?o bezpiecznie uruchomi? od g?ry.
